# Container Build & Run Quickstart

This notebook demonstrates how to build and validate a benchmark container for a single repository commit using the refactored Datasmith orchestration APIs.

## Usage
1. Update the repository parameters below.
2. Point `CONTEXT_REGISTRY_PATH` at a registry JSON that includes your repo/sha (or rely on the default context).
3. Run the notebook top-to-bottom.

> **Note:** Docker, `asv`, and the Datasmith Python package must be available in the current environment.

In [1]:
%load_ext autoreload
%autoreload 2
%cd /mnt/sdd1/atharvas/formulacode/datasmith
import json
from pathlib import Path

import pandas as pd

/mnt/sdd1/atharvas/formulacode/datasmith


In [2]:
results_df = pd.DataFrame([
    json.loads(line)
    for line in Path("scratch/artifacts/pipeflush/symbolic_synthesis/results.jsonl").read_text().splitlines()
    if line != "null"
])
errors = results_df.query("not can_install")
print(errors.shape)
offenders = set(errors["dry_run_log"].str.extract(r"Because ([\w\-]*) was not found").dropna().values.flatten())
offenders.update(
    set(errors["dry_run_log"].str.extract(r"Because there are no versions of ([\w\-]*)").dropna().values.flatten())
)
offenders

(7681, 15)


{'0-18',
 '0-29-21',
 '0-29-30',
 '0-29-32',
 '0-29-33',
 '1-1-4',
 '1-1-5',
 '1-2',
 '1-2-10',
 '1-23-5',
 '1-27-1',
 '1-3-2',
 '2-18-4',
 '2-2',
 '3-0',
 '3-0-0a10',
 '3-0-0a11',
 '3-0-5',
 '59-2-0',
 '6-0-2',
 'arcticdb-ext',
 'c-distances-openmp',
 'cartopy-userconfig',
 'closest-peak-direction-getter',
 'copy-reg',
 'deepchecks-metrics',
 'givens-elimination',
 'jpeg-ls',
 'mio5-utils',
 'mo-pack',
 'mpl-toolkits',
 'optimized-utils',
 'qcs-sdk',
 'sd-common',
 'setup-py',
 'voyager-ext'}

In [3]:
# use can-install data to make tasks.
from datasmith.core.models.task import Task
from datasmith.execution.resolution import analyze_commit

results_df = pd.DataFrame([
    json.loads(line)
    for line in Path("scratch/artifacts/pipeflush/symbolic_synthesis/results.jsonl").read_text().splitlines()
    if line != "null"
])
not_errors = results_df.query("can_install")
resolved_py_version = not_errors["resolution_strategy"].str.extract(r"python=(\d\.\d+)")[0]
not_errors["python_version"] = resolved_py_version
not_errors = not_errors.dropna(subset=["python_version"])
errors = results_df.query("not can_install")

d = errors.groupby("dry_run_log").head(1).to_dict(orient="records")

tasks = [
    Task(
        owner=row["repo_name"].split("/")[0],
        repo=row["repo_name"].split("/")[1],
        sha=row["sha"],
    )
    for row in not_errors.to_dict(orient="records")
]
print(len(tasks))
# for t in random.sample(tasks, 2):
#     task_requirements = analyze_commit(
#     sha=t.sha,
#     repo_name=f"{t.owner}/{t.repo}", bypass_cache=True)
#     if task_requirements['can_install']:
#         continue
#     print(t)
#     print(task_requirements['dry_run_log'])

4253


/tmp/ipykernel_1097584/1278945643.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  not_errors["python_version"] = resolved_py_version


In [4]:
# CONTEXT_REGISTRY_PATH = Path('scratch/artifacts/context_registry_init.json')
OUTPUT_DIR = Path("scratch/notebooks/output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

Output directory: /mnt/sdd1/atharvas/formulacode/datasmith/scratch/notebooks/output


In [5]:
task = tasks[10]
print(task)
task_analysis = analyze_commit(sha=task.sha, repo_name=f"{task.owner}/{task.repo}", bypass_cache=True)
assert task_analysis and task_analysis["can_install"], "Task cannot be installed"
print(task_analysis)

Task(owner='xdslproject', repo='xdsl', sha='4933ebce45e60cd10dd9177f262928710238a37e', commit_date=0.0, env_payload='', python_version='', tag='pkg')
{'sha': '4933ebce45e60cd10dd9177f262928710238a37e', 'repo_name': 'xdslproject/xdsl', 'package_name': 'xdsl', 'package_version': '0.21.1+53.g4933ebce', 'python_version': '3.11', 'build_command': [], 'install_command': [], 'final_dependencies': ['aiohappyeyeballs==2.6.1', 'aiohttp==3.13.0', 'aiosignal==1.4.0', 'anyio==4.11.0', 'asttokens==3.0.0', 'asv==0.6.5', 'asv-runner==0.2.1', 'attrs==25.4.0', 'beautifulsoup4==4.14.2', 'bleach==6.2.0', 'build==1.3.0', 'cfgv==3.4.0', 'click==8.3.0', 'comm==0.2.3', 'coverage==7.10.7', 'debugpy==1.8.17', 'decorator==5.2.1', 'defusedxml==0.7.1', 'distlib==0.4.0', 'docutils==0.22.2', 'executing==2.2.1', 'fastjsonschema==2.21.2', 'filecheck==1.0.0', 'filelock==3.20.0', 'frozenlist==1.8.0', 'h11==0.16.0', 'hypothesis==6.140.3', 'identify==2.6.15', 'idna==3.10', 'immutabledict==4.2.0', 'importlib-metadata==8.7.

In [6]:
from datasmith.notebooks.utils import merge_registries

registries = Path("scratch/").rglob("**/*context_registry*.json")
merged_json = merge_registries(list(registries))

17:08:07 WARNING  simple_useragent.core: Falling back to historic user agent.


In [7]:
import json

from datasmith.docker.context import ContextRegistry
from datasmith.notebooks.utils import update_cr

registry = update_cr(ContextRegistry.deserialize(payload=json.dumps(merged_json)))

In [8]:
from datasmith.agents.build import RE_PY_EXTRACT
from datasmith.docker.context import DockerContext

python_version = ""
if task_analysis.get("resolution_strategy"):
    m = RE_PY_EXTRACT.search(task_analysis["resolution_strategy"])
    if m:
        python_version = m.group(1)
if not python_version:
    python_version = task_analysis.get("python_version", "")
if not python_version and task.python_version:
    python_version = task.python_version

TASK = Task(
    owner=task.owner,
    repo=task.repo,
    sha=task.sha,
    commit_date=task.commit_date,
    python_version=python_version,
    env_payload=json.dumps({"dependencies": task_analysis.get("final_dependencies", "")}) or task.env_payload,
)
pkg_task = TASK.with_tag("pkg")

context = registry.get(pkg_task)
context.building_data == DockerContext().building_data

17:08:28 INFO     datasmith.docker.context: No context found for key 'Task(owner='xdslproject', repo='xdsl', sha='4933ebce45e60cd10dd9177f262928710238a37e', commit_date=0.0, env_payload='{"dependencies": ["aiohappyeyeballs==2.6.1", "aiohttp==3.13.0", "aiosignal==1.4.0", "anyio==4.11.0", "asttokens==3.0.0", "asv==0.6.5", "asv-runner==0.2.1", "attrs==25.4.0", "beautifulsoup4==4.14.2", "bleach==6.2.0", "build==1.3.0", "cfgv==3.4.0", "click==8.3.0", "comm==0.2.3", "coverage==7.10.7", "debugpy==1.8.17", "decorator==5.2.1", "defusedxml==0.7.1", "distlib==0.4.0", "docutils==0.22.2", "executing==2.2.1", "fastjsonschema==2.21.2", "filecheck==1.0.0", "filelock==3.20.0", "frozenlist==1.8.0", "h11==0.16.0", "hypothesis==6.140.3", "identify==2.6.15", "idna==3.10", "immutabledict==4.2.0", "importlib-metadata==8.7.0", "importlib-resources==6.5.2", "iniconfig==2.1.0", "ipykernel==6.30.1", "ipython==9.6.0", "ipython-pygments-lexers==1.1.1", "isort==5.13.2", "itsdangerous==2.2.0", "jedi==0.19.2", "jinja

True

In [9]:
# --- Connect to Docker ---
from datasmith.docker.orchestrator import get_docker_client

client = get_docker_client()
client

In [13]:
# --- Build the package image ---
from datasmith.docker.orchestrator import build_repo_sha_image

print(pkg_task.with_tag("env"))
build_result = build_repo_sha_image(
    client=client,
    docker_ctx=context,
    task=pkg_task.with_tag("env"),
    run_id="notebook-run",
    force=True,
)
build_result

17:14:06 INFO     datasmith.docker.context: Docker image 'xdslproject-xdsl-4933ebce45e60cd10dd9177f262928710238a37e:env' not found locally. Building.


17:14:06 INFO     datasmith.docker.context: $ docker build -t xdslproject-xdsl-4933ebce45e60cd10dd9177f262928710238a37e:env . --build-arg REPO_URL='https://www.github.com/xdslproject/xdsl' --build-arg COMMIT_SHA='4933ebce45e60cd10dd9177f262928710238a37e' --build-arg ENV_PAYLOAD='{"dependencies": ["aiohappyeyeballs==2.6.1", "aiohttp==3.13.0", "aiosignal==1.4.0", "anyio==4.11.0", "asttokens==3.0.0", "asv==0.6.5", "asv-runner==0.2.1", "attrs==25.4.0", "beautifulsoup4==4.14.2", "bleach==6.2.0", "build==1.3.0", "cfgv==3.4.0", "click==8.3.0", "comm==0.2.3", "coverage==7.10.7", "debugpy==1.8.17", "decorator==5.2.1", "defusedxml==0.7.1", "distlib==0.4.0", "docutils==0.22.2", "executing==2.2.1", "fastjsonschema==2.21.2", "filecheck==1.0.0", "filelock==3.20.0", "frozenlist==1.8.0", "h11==0.16.0", "hypothesis==6.140.3", "identify==2.6.15", "idna==3.10", "immutabledict==4.2.0", "importlib-metadata==8.7.0", "importlib-resources==6.5.2", "iniconfig==2.1.0", "ipykernel==6.30.1", "ipython==9.6.0", "ip

Task(owner='xdslproject', repo='xdsl', sha='4933ebce45e60cd10dd9177f262928710238a37e', commit_date=0.0, env_payload='{"dependencies": ["aiohappyeyeballs==2.6.1", "aiohttp==3.13.0", "aiosignal==1.4.0", "anyio==4.11.0", "asttokens==3.0.0", "asv==0.6.5", "asv-runner==0.2.1", "attrs==25.4.0", "beautifulsoup4==4.14.2", "bleach==6.2.0", "build==1.3.0", "cfgv==3.4.0", "click==8.3.0", "comm==0.2.3", "coverage==7.10.7", "debugpy==1.8.17", "decorator==5.2.1", "defusedxml==0.7.1", "distlib==0.4.0", "docutils==0.22.2", "executing==2.2.1", "fastjsonschema==2.21.2", "filecheck==1.0.0", "filelock==3.20.0", "frozenlist==1.8.0", "h11==0.16.0", "hypothesis==6.140.3", "identify==2.6.15", "idna==3.10", "immutabledict==4.2.0", "importlib-metadata==8.7.0", "importlib-resources==6.5.2", "iniconfig==2.1.0", "ipykernel==6.30.1", "ipython==9.6.0", "ipython-pygments-lexers==1.1.1", "isort==5.13.2", "itsdangerous==2.2.0", "jedi==0.19.2", "jinja2==3.1.6", "json5==0.12.1", "jsonschema==4.25.1", "jsonschema-specific

17:14:06 INFO     datasmith.docker.context: Build completed successfully for 'xdslproject-xdsl-4933ebce45e60cd10dd9177f262928710238a37e:env' in 0.2 sec.


BuildResult(ok=True, image_name='xdslproject-xdsl-4933ebce45e60cd10dd9177f262928710238a37e:env', image_id='sha256:7266de84450f9d80da86ad8b9e7fcf3f0ec9f942cb08a32e8de53805c70b5dd2', rc=0, duration_s=0.1963939666748047, stderr_tail='', stdout_tail='Step 1/32 : ARG BASE_IMAGE=buildpack-deps:jammy\nStep 2/32 : ARG PY_VERSION=""                     # passed at build time, used inside stages\nStep 3/32 : FROM ${BASE_IMAGE} AS base\n ---> 7e929bacd45f\nStep 4/32 : ARG PY_VERSION=""\n ---> Using cache\n ---> 7f0458959f9b\nStep 5/32 : RUN apt-get update &&     apt-get install -y --no-install-recommends         jq cmake ninja-build libopenmpi-dev libgeos-dev &&     rm -rf /var/lib/apt/lists/*\n ---> Using cache\n ---> 1acb180e5f63\nStep 6/32 : RUN curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest       | tar -xvj -C /usr/local/bin --strip-components=1 bin/micromamba\n ---> Using cache\n ---> 3054b0905b19\nStep 7/32 : ENV MAMBA_ROOT_PREFIX=/opt/conda     PATH=/opt/conda/bin:$PATH    

In [14]:
# --- Run a quick validation profile ---
import asv

from datasmith.agents.build import ContainerToolExecutor, build_once_with_context
from datasmith.docker.validation import DockerValidator, ValidationConfig

raw_defaults = asv.machine.Machine.get_defaults()  # type: ignore[attr-defined]
machine_defaults = {k: str(v).replace(" ", "_") for k, v in raw_defaults.items()}

config = ValidationConfig(output_dir=OUTPUT_DIR, build_timeout=1800, run_timeout=600, tail_chars=4000)
validator = DockerValidator(
    client=client,
    context_registry=registry,
    machine_defaults=machine_defaults,
    config=config,
)

tool_exec = ContainerToolExecutor(
    docker_client=client,
    image_name=task.with_tag("env").get_image_name(),
    container_name=task.with_tag("env").get_container_name(),
    workdir="/workspace/repo/",
    run_labels={},
)
# run_result = validator.validate_task(TASK.with_tag("run"), run_labels={})
# run_result

r = validator.build_and_validate(
    task=TASK.with_tag("run"),
    context=context,
    run_labels={},
    build_once_fn=build_once_with_context,
)
print(r)

17:14:11 INFO     datasmith.docker.validation: build_and_validate: building image 'xdslproject-xdsl-4933ebce45e60cd10dd9177f262928710238a37e:run'
17:14:11 INFO     datasmith.docker.context: Docker image 'xdslproject-xdsl-4933ebce45e60cd10dd9177f262928710238a37e:run' found locally (skip build).
17:14:11 INFO     datasmith.agents.build: build_once_with_context: result ok=True rc=0 duration=0.0s (stderr_tail_len=0, stdout_tail_len=0)
17:14:11 INFO     datasmith.docker.validation: build_and_validate: build ok; verifying profile+tests before recording attempt
17:14:33 WARNING  datasmith.docker.validation: build_and_validate: test validation failed


BuildResult(ok=False, image_name='xdslproject-xdsl-4933ebce45e60cd10dd9177f262928710238a37e:run', image_id='sha256:35065fa090714270920aceec6a22c1749adf3f84b943b91b0d0f212409f682eb', rc=1, duration_s=0.006772756576538086, stderr_tail="+ cd /workspace/repo\n+ set +ux\n+ '[' 0 -gt 0 ']'\n+ '[' -n 4933ebce45e60cd10dd9177f262928710238a37e ']'\n+ FORMULACODE_BASE_COMMIT=4933ebce45e60cd10dd9177f262928710238a37e\n+ reset_repo_state 4933ebce45e60cd10dd9177f262928710238a37e\n+ local COMMIT_SHA=4933ebce45e60cd10dd9177f262928710238a37e\n++ git remote -v\n++ grep '(fetch)'\n++ awk '{print $2}'\n+ URL=https://www.github.com/xdslproject/xdsl\n+ [[ https://www.github.com/xdslproject/xdsl =~ ^(https://)?(www\\.)?github\\.com/dask/dask(\\.git)?$ ]]\n+ [[ https://www.github.com/xdslproject/xdsl =~ ^(https://)?(www\\.)?github\\.com/dask/distributed(\\.git)?$ ]]\n+ [[ https://www.github.com/xdslproject/xdsl =~ ^(https://)?(www\\.)?github\\.com/joblib/joblib(\\.git)?$ ]]\n+ [[ https://www.github.com/xdslpro

In [ ]:
# synthesize context if build_result errors out.
from datasmith.agents.build import BuildScriptProgram, synthesize_script

program = BuildScriptProgram()

script = synthesize_script(
    program,
    task,
    context.building_data,
    stderr_tail=r.stderr_tail,
    stdout_tail=r.stdout_tail,
    failure_more=r.rc,
    tool_exec=tool_exec,
    max_steps=5,
)

17:18:13 INFO     datasmith.agents.build: synthesize_script: task=xdslproject/xdsl@4933ebce45e60cd10dd9177f262928710238a37e, last_script=present
17:18:13 INFO     datasmith.agents.build: DSPy: synthesizing build script for xdslproject/xdsl@4933ebce45e60cd10dd9177f262928710238a37e (stderr_len=1708, stdout_len=3901, has_last=True, failure=1)
17:18:13 ERROR    datasmith.agents.build: synthesize_script: error
Traceback (most recent call last):
  File "/mnt/sdd1/atharvas/formulacode/datasmith/src/datasmith/agents/build.py", line 268, in synthesize_script
    result = program(
             ^^^^^^^^
  File "/mnt/sdd1/atharvas/formulacode/datasmith/.venv/lib/python3.12/site-packages/dspy/utils/callback.py", line 326, in sync_wrapper
    return fn(instance, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/sdd1/atharvas/formulacode/datasmith/.venv/lib/python3.12/site-packages/dspy/primitives/module.py", line 78, in __call__
    return self.forward(*args, **kwargs)
         

In [ ]:
# --- Optional cleanup ---
from datasmith.docker.orchestrator import generate_run_labels

labels = generate_run_labels(TASK, run_id="notebook-run")
print("Labels for this run:", labels)
print("Use docker CLI to prune images/containers if desired.")

ImportError: cannot import name 'generate_run_labels' from 'datasmith.docker.orchestrator' (/mnt/sdd1/atharvas/formulacode/datasmith/src/datasmith/docker/orchestrator.py)